# MPPP — 03 COLMAP alignment (Navcam, optionally + Mastcam-Z 34 mm)

From PDS products to a COLMAP database and a refined, prior-aligned reconstruction. Pick a named site (`SITES`: Belva 748–815, Rockytop 461–530, Three Forks 684–692; add your own) or edit the sol range. With `INCLUDE_ZCAM34 = True` the Mastcam-Z 34 mm frames (`ZL0`/`ZR0`, `_034`) of the same sols join the block: one COLMAP camera per eye (`ZL034`, `ZR034`), initialised from the median label CAHVOR model (or the rough `m20_cmods` XML) and refined like the Navcam cameras. Mastcam-Z left and right exposures have different clocks, so they are separate frames without a rig constraint (experimental).

1. **Select**: Navcam `NLF`/`NRF`, NCAM sequences, one product per exposure (the largest, then the highest version).
2. **Process** with MPPP: 8-bit linear PNG, masks, calibrated intrinsics, CAHV + waypoint poses, frames padded to the detector frame.
3. **Project** (`SfmProject`): one COLMAP camera per eye at full resolution from `mppp/data/m20_cmods/M2020_N{L,R}0_frame.xml` (p1 = p2 = b1 = b2 = 0); half- and quarter-resolution keypoints are scaled to it. One rig: left is the reference, the right camera's offset is the median CAHV left→right pose.
4. **Features** at native resolution with masks → `features.db` (reused if present). **Database** → `database.db`, rebuilt on every run (cameras, rig, frames = exposures, full-resolution keypoints, waypoint position priors).
5. **Matching**: `exhaustive` (baseline; use a GPU) or `prior_pairs` (only pairs the priors say overlap; quick / CPU).
6. **Reconstruct**: triangulation from the CAHV poses (two-view tie points kept), then a weighted bundle adjustment (σ in native pixels, Cauchy loss, waypoint priors). Refined: f, principal point and k1–k3 per camera, the right camera's rig orientation, the frame poses; the CAHV stereo baseline is held (it sets the scale).
7. **Alignment health**: tie-point population, reprojection error, camera-model change, stereo-rig stability, pose change against the priors. **Check this before using the result.**
8. Export `error_input/` (native-pixel text model, stations, poses, residuals) — and, under development, the error analysis.

Needs pip `pycolmap` 4.2 and `pyceres` 2.6 in this kernel (`pip install -e .[sfm]`).

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import sys, json
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
print("mppp", mppp.__version__, "from", Path(mppp.__file__).parent)
import pycolmap, pyceres
from mppp.select import find_imgs
from mppp.sfm import (SfmProject, select_best_products, extract_features, build_database, prior_overlap_pairs,
                      match, reconstruct, export_for_error, check_gpu_python, check_ba_environment)
from mppp.sfm.health import assess_alignment, health_table, write_health
print("MPPP", mppp.__version__, "| pycolmap", pycolmap.__version__, "(CUDA:", pycolmap.has_cuda, ") | pyceres", pyceres.__version__)
check_ba_environment()        # stops here if this kernel cannot run the weighted BA (e.g. it is the conda CUDA env)

# Optional GPU for feature extraction and matching (Windows): the python.exe of a second environment with the
# conda-forge CUDA pycolmap of the same COLMAP version (README, "GPU on Windows").  None = run here on the CPU.
GPU_PY = None                 # e.g. r"C:\Users\<you>\AppData\Local\miniconda3\envs\mppp_gpu\python.exe"
if GPU_PY:
    print("GPU env:", check_gpu_python(GPU_PY))

## 1–2. Select and process

In [ ]:
PDS_DIR  = Path("D:/data/m2020")

# named test sites: sol range of the Navcam (+ Mastcam-Z) block; add your own
SITES = {
    "belva":      (748, 815),
    "rockytop":   (461, 530),
    "threeforks": (684, 692),
}
SITE = "belva"
SOLS = SITES[SITE]

INCLUDE_ZCAM34  = False        # True: add the Mastcam-Z 34 mm frames (ZL0/ZR0 "_034") of the same sols
ZCAM_INTRINSICS = "label"      # initial Mastcam-Z cameras: "label" (median of the label CAHVOR models) or "xml"
# processed images + COLMAP project go here (a separate folder when Mastcam-Z is included)
WORK = Path("D:/scapes") / (f"{SITE}_colmap_nav_zcam34" if INCLUDE_ZCAM34 else f"{SITE}_colmap")

paths = find_imgs(PDS_DIR, ["NLF", "NRF"], sol_range=SOLS)
prefixes = ("NCAM",)
if INCLUDE_ZCAM34:
    paths += find_imgs(PDS_DIR, ["ZL0", "ZR0"], sol_range=SOLS, sequ_id="_034")
    prefixes += ("ZCAM",)
paths, sel = select_best_products(paths, sequence_prefix=prefixes)      # one product per exposure
print(SITE, SOLS, "->", WORK)
print(json.dumps(sel, indent=1)); print(len(paths), "products")

In [ ]:
cfg = mppp.default_config()
cfg["export"].update(formats=["PNG8"], store_mask_in_alpha=False, write_mask_files=True, write_colmap=False)
# mask model: the released "mppp_mask_v1" by default; or cfg["masking"]["checkpoint"] = "<path to your checkpoint>"
PROCESSED = WORK / "processed"
# Frames unsuitable for the analysis: after a first run, delete them from PROCESSED / "images_png8" and set
# KEEP_ONLY_REMAINING = True: only the images left there are processed again, the manifest is rebuilt from them,
# and so is the COLMAP project below (nothing is deleted).
KEEP_ONLY_REMAINING = False
manifest_path = PROCESSED / f"mppp_manifest_{mppp.VERSION_TAG}.json"
if manifest_path.is_file() and not KEEP_ONLY_REMAINING:
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = mppp.process_images(paths, PROCESSED, cfg, mppp.load_waypoints(), provenance=sel,
                                   only_existing="PNG8" if KEEP_ONLY_REMAINING else None)
print(manifest["n_processed"], "processed,", len(manifest["failed"]), "failed")
if manifest.get("only_existing"):
    print("kept", manifest["only_existing"]["n_kept"], "of", manifest["only_existing"]["n_selected"], "selected;",
          manifest["only_existing"]["n_removed"], "removed")

## 3–4. Project, features, database

In [ ]:
PROJECT = WORK / "colmap"
if (PROJECT / "project.json").is_file() and not KEEP_ONLY_REMAINING:      # rebuilt from the new manifest
    proj = SfmProject.load(PROJECT)
else:
    proj = SfmProject.create(manifest["images"], PROCESSED, PROJECT, prior_sigma_m=(1.0, 1.0, 1.0),
                            zcam_intrinsics=ZCAM_INTRINSICS)
print(len(proj.images), "images;", {k: round(v["baseline_m"], 4) for k, v in proj.rig.items()}, "m baseline;",
      {k: v["n_pairs"] for k, v in proj.rig.items()}, "CAHV pairs")
print({k: [round(x, 3) for x in c["params"][:5]] for k, c in proj.cameras.items()})
from collections import Counter
print("images per camera:", dict(Counter(r["instrument"] for r in proj.images)))

In [ ]:
# SIFT at native resolution with masks -> features.db (reused on reruns; overwrite=True to redo).
# COLMAP GUI/CLI alternative: colmap feature_extractor --database_path features.db --image_path images
#     --ImageReader.mask_path masks --SiftExtraction.max_num_features 8192   (check option names with -h)
extract_features(proj, max_num_features=8192, use_gpu=True, python=GPU_PY)
print(build_database(proj))          # replaces database.db (and any matches in it) on every run

## 5. Matching

In [ ]:
MATCH_MODE = "exhaustive"            # baseline; "prior_pairs" for a quick / CPU run
if MATCH_MODE == "prior_pairs":
    pairs = prior_overlap_pairs(proj, min_overlap=0.05)
    print(len(pairs), "prior pairs")
print(match(proj, mode=MATCH_MODE, use_gpu=True, python=GPU_PY))

## 6. Reconstruct and refine

In [ ]:
rec = reconstruct(proj, sigma_px=0.5, refine_rig="rotation",
                  min_track_length=2,        # two-view tie points (e.g. seen only by one stereo pair) are kept
                  min_tri_angle_deg=1.5)     # 1.5 deg keeps stereo-only points out to ~16 m at the 0.42 m baseline
summary = export_for_error(proj, rec)
print(json.dumps({k: summary[k] for k in ("tracks", "residual_rms_native_px", "registered_images", "project_images",
                                           "prior_offset_median_m", "station_components")}, indent=1))

## 7. Alignment health — check before any analysis
Each check has provisional warn/fail thresholds (set a priori, not validated limits; pass `thresholds={...}` to change them). The report is written to `<project>/health/` (`health.json`, `health.md`, `health.png`).

| section | checks |
|---|---|
| registration | images registered; frames held at their prior (< 30 observations) |
| tie points | points, track-length population (two-view share), observations per image, cross-station ties, tied station blocks |
| reprojection | median / 95th percentile residual (native px), left vs right eye, outlier images, residual growth towards the image edge |
| camera model | change of f, principal point, and the image displacement of the same ray |
| stereo rig | right-camera rotation vs the CAHV rig; spread of the CAHV pairs |
| poses | refined − prior centre per station, spread within a station, attitude change, scale vs the priors |

In [ ]:
report = assess_alignment(proj, rec)
print(health_table(report))
files = write_health(report, proj.root / "health", rec, proj)
from IPython.display import Image, display
display(Image(filename=files["png"]))
if report["verdict"] == "fail":
    print("\nFAIL: fix the alignment (matching, priors, masks, thresholds) before any error analysis.")

In [ ]:
# plan view: prior -> refined camera centre per station (arrows exaggerated)
import numpy as np, csv
import matplotlib.pyplot as plt
rows = [r for r in csv.DictReader(open(proj.root / "error_input" / "poses.csv")) if int(r["observations"]) >= 30]
st = {}
for r in rows:
    st.setdefault(r["station"], []).append([float(r[k]) for k in ("E_m", "N_m", "dE_m", "dN_m")])
fig, ax = plt.subplots(figsize=(7, 6))
k = 100.0
for name, v in sorted(st.items()):
    E, N, dE, dN = np.median(np.array(v), axis=0)
    ax.annotate("", (E, N), (E - k * dE, N - k * dN), arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.plot(E, N, "o", ms=4); ax.text(E, N, " " + name, fontsize=7)
ax.set_aspect("equal"); ax.set_title(f"prior -> refined station centre (x{k:g})"); ax.set_xlabel("E [m]"); ax.set_ylabel("N [m]")

## 8. Error analysis (`mppp.error`, under development)
Only meaningful for an alignment that passes the health checks.

In [ ]:
assert report["verdict"] != "fail", "alignment health is FAIL - see section 7"
from mppp.error.colmap import read_colmap, calibrate_eps_from_residuals
model = read_colmap(str(proj.root / "error_input" / "native"))
stations = {r["name"]: r["station"] for r in csv.DictReader(open(proj.root / "error_input" / "stations.csv"))}
model.assign_stations(stations)
print(calibrate_eps_from_residuals(model))